In [7]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import gcamreader

In [8]:
# ----------------------------
# Config
# ----------------------------
DBPATH    = "../output/"
DBFILE    = "database_basexdb_korea_2035"
SCENARIOS = ["Current-Policies-Med", "Enhanced-Ambition-Med"]

# Service subsectors to include (passenger + road freight core groups)
SERVICE_SECTORS_ALL   = ["Bus", "Car", "Large Car and Truck", "Medium truck"]
UNITS_PASSENGER       = "million pass-km"
UNITS_FREIGHT         = "million ton-km"

# Colors per scenario
COLOR_MAP = {
    SCENARIOS[0]:      "#636EFA",   # blue
    SCENARIOS[1]:   "#00CC96",   # green
}

# Nicely formatted scenario names for legends
PRETTY = {
    SCENARIOS[0]: "Current Policies",
    SCENARIOS[1]: "Enhanced Ambition",
}

In [9]:
# ----------------------------
# Helpers
# ----------------------------
def connect_and_load_queries():
    conn = gcamreader.LocalDBConn(DBPATH, DBFILE)
    queries = gcamreader.parse_batch_query(os.path.join("..", "output", "queries", "Main_queries.xml"))
    return conn, queries

def run_query(conn, q, scenarios=SCENARIOS, region="South Korea"):
    """Run a GCAM batch query and normalize scenario column."""
    df = conn.runQuery(q, scenarios=scenarios, regions=[region])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def add_zev_flag(df, zev_tech=("BEV", "FCEV")):
    """Add a binary ZEV flag per technology."""
    df = df.copy()
    df["ZEV"] = df["technology"].isin(zev_tech).astype(int)
    return df

def zev_share_and_service(df, service_sectors, min_year=2020):
    """
    From a service-activity query result:
      - Return ZEV share (%) of service (by Year/Units)
      - Return ZEV absolute service in *billions* of the same units
    """
    df = df[df["subsector"].isin(service_sectors)].copy()
    df = add_zev_flag(df)

    # totals and zev-only (≥ min_year)
    mask_year = df["Year"] >= min_year
    tot = (df[mask_year]
           .groupby(["scenario", "Year", "Units"], as_index=False)["value"].sum()
           .rename(columns={"value": "total"}))

    zev = (df[mask_year & (df["ZEV"] == 1)]
           .groupby(["scenario", "Year", "Units"], as_index=False)["value"].sum()
           .rename(columns={"value": "zev"}))

    out = tot.merge(zev, on=["scenario", "Year", "Units"], how="left").fillna({"zev": 0})
    out["share_percent"] = (out["zev"] / out["total"]) * 100.0
    # convert service to billions for nicer scale
    out["zev_billion"] = out["zev"] / 1000.0
    return out

def add_dual_series(fig, df, units_filter, scenario, left_name, right_name):
    """Add (left) ZEV share line and (right) ZEV service bars for one scenario."""
    pretty = PRETTY.get(scenario, scenario)
    clr    = COLOR_MAP.get(scenario, "grey")

    sub = df[df["Units"] == units_filter]

    # Left y (%): share
    fig.add_trace(go.Scatter(
        x=sub["Year"], y=sub["share_percent"],
        name=f"{left_name} in {pretty}",
        mode="lines+markers",
        line=dict(color=clr),
        yaxis="y1"
    ))

    # Right y (billions): ZEV service
    fig.add_trace(go.Bar(
        x=sub["Year"], y=sub["zev_billion"],
        name=f"{right_name} in {pretty}",
        marker_color=clr, opacity=0.5,
        yaxis="y2"
    ))

def base_layout(fig, y2_title, width=600, height=450, legend_pos="h"):
    """Apply consistent layout across figures."""
    # legend defaults
    if legend_pos == "h":
        legend_cfg = dict(x=0.5, y=-0.2, xanchor="center", orientation="h")
    else:
        legend_cfg = dict(
            x=0.02, y=0.98, xanchor="left", yanchor="top",
            bgcolor="rgba(255,255,255,0.6)", bordercolor="lightgray", borderwidth=1,
            orientation="v"
        )

    fig.update_layout(
        barmode="group",
        legend=legend_cfg,
        plot_bgcolor="white",
        paper_bgcolor="white",
        height=height,
        width=width,
        font=dict(size=16)
    )

    fig.update_layout(
        xaxis=dict(
            title="Year",
            showticklabels=True,
            ticks="outside",
            ticklen=6, tickwidth=2, tickcolor="black",
            showline=True, linecolor="black", linewidth=1,
            mirror=True
        ),
        yaxis=dict(
            title=dict(text="New Sales Share (%)", font=dict(color="black")),
            showticklabels=True,
            ticks="outside",
            ticklen=6, tickwidth=2, tickcolor="black",
            showline=True, linecolor="black", linewidth=1,
            dtick=10, mirror=True
        ),
        yaxis2=dict(
            title=dict(text=y2_title, font=dict(color="black")),
            tickcolor="black", ticklen=6, tickwidth=2, ticks="outside",
            overlaying="y", side="right",
            showline=True, linecolor="black", linewidth=1
        )
    )
    return fig

In [10]:
conn, queries = connect_and_load_queries()

# Query indices:
#   159 → (example) road passenger service by technology
#   161 → (example) road freight service by technology
# Adjust if your batch file indices differ.
Q_PASS = queries[159]
Q_FRT  = queries[161]

# Run queries
df_pass = run_query(conn, Q_PASS, scenarios=SCENARIOS)
df_frt  = run_query(conn, Q_FRT,  scenarios=SCENARIOS)

# Compute ZEV share & service
df_pass_agg = zev_share_and_service(df_pass, SERVICE_SECTORS_ALL, min_year=2020)
df_frt_agg  = zev_share_and_service(df_frt,  SERVICE_SECTORS_ALL, min_year=2020)

Database scenarios: Current-Policies-Med, Enhanced-Ambition-Med, Current-Policies-High, Current-Policies-Low, Enhanced-Ambition-High, Enhanced-Ambition-Low


In [17]:
# ----------------------------
# Passenger figure
# ----------------------------
fig_passenger = go.Figure()
for scen in SCENARIOS:
    add_dual_series(
        fig_passenger,
        df_pass_agg[df_pass_agg["scenario"] == scen],
        UNITS_PASSENGER,
        scenario=scen,
        left_name="New Sales",
        right_name="ZEV Service"
    )
base_layout(fig_passenger, y2_title="billion pass–km", width=620, height=460, legend_pos="h")
pio.write_image(fig_passenger, "./fig/pass.png", width=620, height=460, scale=2)

fig_passenger.update_layout(
    yaxis=dict(
        title=dict(text="New Sales Share (%)", font=dict(color="black")),
        range=[0, 80],  # <- yaxis1 범위 지정
        showticklabels=True,
        ticks="outside",
        ticklen=6, tickwidth=2, tickcolor="black",
        showline=True, linecolor="black", linewidth=1,
        dtick=10, mirror=True
    ),
    yaxis2=dict(
        title=dict(text="billion pass–km", font=dict(color="black")),
        tickcolor="black", ticklen=6, tickwidth=2, ticks="outside",
        overlaying="y", side="right",
        showline=True, linecolor="black", linewidth=1
    ),
    legend=dict(
        x=0.02,       # 왼쪽 여백 (0=완전 왼쪽)
        y=0.98,       # 위쪽 여백 (1=완전 위)
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="lightgray",
        borderwidth=1,
        orientation="v"  # 세로 정렬
    )
)
fig_passenger.show()

In [16]:
# ----------------------------
# Freight figure
# ----------------------------
fig_freight = go.Figure()
for scen in SCENARIOS:
    add_dual_series(
        fig_freight,
        df_frt_agg[df_frt_agg["scenario"] == scen],
        UNITS_FREIGHT,
        scenario=scen,
        left_name="New Sales",
        right_name="ZEV Service"
    )
# Vertical legend works better here due to bar stacking width
base_layout(fig_freight, y2_title="billion ton–km", width=620, height=460, legend_pos="v")
pio.write_image(fig_freight, "./fig/freight.png", width=620, height=460, scale=2)

fig_freight.update_layout(
    yaxis=dict(
        title=dict(text="New Sales Share (%)", font=dict(color="black")),
        range=[0, 80],  # <- yaxis1 범위 지정
        showticklabels=True,
        ticks="outside",
        ticklen=6, tickwidth=2, tickcolor="black",
        showline=True, linecolor="black", linewidth=1,
        dtick=10, mirror=True
    ),
    yaxis2=dict(
        title=dict(text="billion pass–km", font=dict(color="black")),
        tickcolor="black", ticklen=6, tickwidth=2, ticks="outside",
        overlaying="y", side="right",
        showline=True, linecolor="black", linewidth=1
    ),
    legend=dict(
        x=0.02,       # 왼쪽 여백 (0=완전 왼쪽)
        y=0.98,       # 위쪽 여백 (1=완전 위)
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="lightgray",
        borderwidth=1,
        orientation="v"  # 세로 정렬
    )
)

fig_freight.show()